In [3]:
"""
Role: Computer Vision Engineer

Task:
Build a real-time face detection application using Python.

Tech Stack:
- Python
- OpenCV
- Haar Cascade (initially), with option to extend to DNN-based detection later

Requirements:
- Capture video from webcam
- Detect faces in real-time
- Draw bounding boxes around detected faces
- Display live video feed with detections
- Show FPS (frames per second) on screen
- Allow user to exit with 'q' key

Architecture Constraints:
- Write clean, modular code
- Separate logic into functions:
  - initialize_camera()
  - load_model()
  - detect_faces()
  - draw_bounding_boxes()
  - main_loop()
- Avoid writing everything in one block

Performance Requirements:
- Optimize for real-time performance
- Use grayscale conversion for faster detection
- Resize frames if needed for speed

Error Handling:
- Handle webcam access failure
- Handle model loading failure

Optional Enhancements (structure code to support these later):
- Face recognition (future extension)
- Emotion detection
- Save detected faces as images
- Switch between Haar Cascade and DNN model

UI Requirements:
- Display:
  - Bounding boxes
  - Number of faces detected
  - FPS counter

Output Requirements:
- Provide complete working Python code
- No explanations
- Code must be ready to run

Instruction:
Start with Haar Cascade implementation only.
"""
import cv2
import time
import sys

def initialize_camera(camera_id=0):
    """Initialize and return video capture object."""
    cap = cv2.VideoCapture(camera_id)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        sys.exit(1)
    return cap

def load_model(model_path=cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'):
    """Load Haar Cascade classifier for face detection."""
    face_cascade = cv2.CascadeClassifier(model_path)
    if face_cascade.empty():
        print("Error: Could not load Haar Cascade model.")
        sys.exit(1)
    return face_cascade

def detect_faces(face_cascade, frame, scale_factor=1.1, min_neighbors=5):
    """Detect faces in a frame using Haar Cascade."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=(30, 30)
    )
    return faces, gray

def draw_bounding_boxes(frame, faces):
    """Draw bounding boxes around detected faces and return face count."""
    for (x, y, w, h) in faces:
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(frame, 'Face', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 
                    0.5, (0, 255, 0), 2)
    return len(faces)

def display_fps(frame, fps):
    """Display FPS on the frame."""
    cv2.putText(frame, f'FPS: {fps:.1f}', (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

def display_face_count(frame, face_count):
    """Display number of faces detected on the frame."""
    cv2.putText(frame, f'Faces: {face_count}', (10, 60), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

def main_loop():
    """Main application loop for real-time face detection."""
    cap = initialize_camera()
    face_cascade = load_model()
    
    prev_time = time.time()
    fps = 0.0
    
    print("Press 'q' to exit")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break
        
        # Optional: Resize frame for faster processing
        # frame = cv2.resize(frame, (640, 480))
        
        # Detect faces
        faces, gray = detect_faces(face_cascade, frame)
        
        # Draw bounding boxes and get count
        face_count = draw_bounding_boxes(frame, faces)
        
        # Calculate FPS
        current_time = time.time()
        fps = 1.0 / (current_time - prev_time)
        prev_time = current_time
        
        # Display information
        display_fps(frame, fps)
        display_face_count(frame, face_count)
        
        # Show the frame
        cv2.imshow('Real-Time Face Detection', frame)
        
        # Exit on 'q' key
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Cleanup
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    try:
        main_loop()
    except KeyboardInterrupt:
        print("\nApplication stopped by user.")
        cv2.destroyAllWindows()
        sys.exit(0)

Press 'q' to exit


In [2]:
import cv2
import time
import sys
from collections import deque

def initialize_camera(camera_id=0, width=640, height=480):
    """Initialize and return video capture object with optimized settings."""
    cap = cv2.VideoCapture(camera_id, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("Error: Could not open webcam.")
        sys.exit(1)
    
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
    cap.set(cv2.CAP_PROP_FPS, 30)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    
    return cap

def load_model(model_path=cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'):
    """Load Haar Cascade classifier for face detection."""
    face_cascade = cv2.CascadeClassifier(model_path)
    if face_cascade.empty():
        print("Error: Could not load Haar Cascade model.")
        sys.exit(1)
    return face_cascade

def detect_faces_optimized(face_cascade, gray, scale_factor=1.05, min_neighbors=3):
    """Detect faces with optimized parameters for speed."""
    if gray.shape[0] < 30 or gray.shape[1] < 30:
        return []
    
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=(40, 40),
        flags=cv2.CASCADE_SCALE_IMAGE
    )
    return faces

def process_frame_for_detection(frame, skip_frames=1, frame_count=0):
    """Process frame with optional frame skipping for CPU optimization."""
    if frame_count % skip_frames != 0:
        return None, frame_count + 1
    
    height, width = frame.shape[:2]
    if width > 480:
        scale = 480 / width
        new_width = 480
        new_height = int(height * scale)
        detection_frame = cv2.resize(frame, (new_width, new_height))
    else:
        detection_frame = frame
    
    return detection_frame, frame_count + 1

def draw_bounding_boxes_optimized(frame, faces, original_scale=1.0):
    """Draw bounding boxes with scaling adjustment."""
    face_count = 0
    for (x, y, w, h) in faces:
        if original_scale != 1.0:
            x = int(x / original_scale)
            y = int(y / original_scale)
            w = int(w / original_scale)
            h = int(h / original_scale)
        
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        face_count += 1
    
    return face_count

def calculate_fps_optimized(fps_history, current_time, prev_time, history_size=10):
    """Calculate FPS using moving average for stability."""
    current_fps = 1.0 / (current_time - prev_time)
    fps_history.append(current_fps)
    if len(fps_history) > history_size:
        fps_history.popleft()
    return sum(fps_history) / len(fps_history)

def display_info_optimized(frame, fps, face_count):
    """Display FPS and face count with minimal draw calls."""
    info_text = f'FPS: {fps:.1f} | Faces: {face_count}'
    cv2.putText(frame, info_text, (10, 25), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)
    
    if fps < 15:
        cv2.putText(frame, 'High CPU - Reduce resolution', (10, 50), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1, cv2.LINE_AA)

def main_loop():
    """Optimized main application loop for real-time face detection."""
    cap = initialize_camera(width=640, height=480)
    face_cascade = load_model()
    
    prev_time = time.time()
    fps_history = deque(maxlen=10)
    frame_counter = 0
    skip_frames = 1
    
    # Store last faces as list of tuples, not numpy array
    last_faces = []
    
    print("Press 'q' to exit")
    print("Controls:")
    print("  '+' - Decrease frame skip (better accuracy)")
    print("  '-' - Increase frame skip (better performance)")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break
        
        # Process frame for detection with optional frame skipping
        detection_frame, frame_counter = process_frame_for_detection(
            frame, skip_frames, frame_counter
        )
        
        if detection_frame is not None:
            # Convert to grayscale
            gray = cv2.cvtColor(detection_frame, cv2.COLOR_BGR2GRAY)
            
            # Apply ROI optimization if faces were detected before
            if len(last_faces) > 0 and frame_counter % 2 == 0:
                roi_faces = []
                scale_x = detection_frame.shape[1] / frame.shape[1]
                scale_y = detection_frame.shape[0] / frame.shape[0]
                
                for (x, y, w, h) in last_faces:
                    x_roi = max(0, int(x * scale_x) - 30)
                    y_roi = max(0, int(y * scale_y) - 30)
                    w_roi = min(detection_frame.shape[1] - x_roi, int(w * scale_x) + 60)
                    h_roi = min(detection_frame.shape[0] - y_roi, int(h * scale_y) + 60)
                    
                    if w_roi > 20 and h_roi > 20:
                        roi = gray[y_roi:y_roi+h_roi, x_roi:x_roi+w_roi]
                        faces_roi = face_cascade.detectMultiScale(roi, 1.05, 3, minSize=(30, 30))
                        for (fx, fy, fw, fh) in faces_roi:
                            roi_faces.append((x_roi + fx, y_roi + fy, fw, fh))
                
                if len(roi_faces) > 0:
                    faces = roi_faces
                else:
                    faces = detect_faces_optimized(face_cascade, gray)
            else:
                faces = detect_faces_optimized(face_cascade, gray)
            
            # Store faces for ROI optimization (convert to list of tuples)
            if len(faces) > 0:
                last_faces = [(int(x), int(y), int(w), int(h)) for (x, y, w, h) in faces]
            
            # Calculate scaling factor for drawing
            scale_factor = frame.shape[1] / detection_frame.shape[1] if detection_frame is not frame else 1.0
            
            # Draw bounding boxes
            face_count = draw_bounding_boxes_optimized(frame, faces, scale_factor)
        else:
            # If frame is skipped, use last known face positions
            face_count = draw_bounding_boxes_optimized(frame, last_faces, 1.0)
        
        # Calculate FPS
        current_time = time.time()
        fps = calculate_fps_optimized(fps_history, current_time, prev_time)
        prev_time = current_time
        
        # Display information
        display_info_optimized(frame, fps, face_count)
        
        # Show frame
        cv2.imshow('Real-Time Face Detection', frame)
        
        # Handle keyboard input
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('+') and skip_frames > 1:
            skip_frames -= 1
            print(f"Frame skipping reduced to {skip_frames} (better accuracy)")
        elif key == ord('-'):
            skip_frames += 1
            print(f"Frame skipping increased to {skip_frames} (better performance)")
    
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    try:
        main_loop()
    except KeyboardInterrupt:
        print("\nApplication stopped by user.")
        cv2.destroyAllWindows()
        sys.exit(0)

Press 'q' to exit
Controls:
  '+' - Decrease frame skip (better accuracy)
  '-' - Increase frame skip (better performance)


In [4]:
import cv2
import time
import sys
import os


# ─────────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────────
CONFIG = {
    "camera_index": 0,
    "frame_width": 640,
    "frame_height": 480,
    "scale_factor": 1.1,
    "min_neighbors": 5,
    "min_face_size": (30, 30),
    "detection_scale": 0.5,       # resize frame before detection for speed
    "box_color": (0, 255, 0),     # green bounding boxes
    "box_thickness": 2,
    "font": cv2.FONT_HERSHEY_SIMPLEX,
    "fps_smoothing": 10,          # rolling average over N frames
}

# Model backend — swap to "dnn" later without changing the rest of the code
MODEL_BACKEND = "haar"


# ─────────────────────────────────────────────
#  Camera
# ─────────────────────────────────────────────
def initialize_camera(index: int = 0) -> cv2.VideoCapture:
    """Open the webcam and apply preferred resolution settings."""
    cap = cv2.VideoCapture(index)
    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open camera at index {index}. "
            "Check that a webcam is connected and not in use by another app."
        )

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, CONFIG["frame_width"])
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CONFIG["frame_height"])
    cap.set(cv2.CAP_PROP_FPS, 30)

    # Verify the camera actually delivers frames
    ok, test_frame = cap.read()
    if not ok or test_frame is None:
        cap.release()
        raise RuntimeError("Camera opened but failed to deliver a frame.")

    actual_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    actual_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"[Camera] Opened index={index}  resolution={actual_w}x{actual_h}")
    return cap


# ─────────────────────────────────────────────
#  Model loading (Haar Cascade)
# ─────────────────────────────────────────────
def load_model(backend: str = "haar"):
    """
    Load the face-detection model.

    Returns a dict so callers are backend-agnostic:
        { "backend": str, "detector": <model object> }

    Extend this function with a "dnn" branch later:
        elif backend == "dnn":
            net = cv2.dnn.readNetFromCaffe(prototxt, caffemodel)
            return {"backend": "dnn", "detector": net}
    """
    if backend == "haar":
        cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        if not os.path.isfile(cascade_path):
            raise FileNotFoundError(
                f"Haar cascade file not found: {cascade_path}\n"
                "Re-install opencv-python: pip install opencv-python"
            )
        detector = cv2.CascadeClassifier(cascade_path)
        if detector.empty():
            raise RuntimeError("CascadeClassifier loaded but is empty — file may be corrupt.")
        print(f"[Model] Loaded Haar cascade from {cascade_path}")
        return {"backend": "haar", "detector": detector}

    # ── Placeholder for future DNN backend ──────────────────────────────────
    # elif backend == "dnn":
    #     prototxt     = "deploy.prototxt"
    #     caffemodel   = "res10_300x300_ssd_iter_140000.caffemodel"
    #     net = cv2.dnn.readNetFromCaffe(prototxt, caffemodel)
    #     return {"backend": "dnn", "detector": net}
    # ────────────────────────────────────────────────────────────────────────

    raise ValueError(f"Unknown model backend: '{backend}'. Choose 'haar' or 'dnn'.")


# ─────────────────────────────────────────────
#  Detection
# ─────────────────────────────────────────────
def detect_faces(frame, model: dict) -> list:
    """
    Detect faces in *frame* using the loaded model.

    Returns a list of (x, y, w, h) tuples in **original frame** coordinates.

    The frame is downscaled before detection for speed, then coordinates are
    mapped back — keeping draw_bounding_boxes() unaware of the optimisation.
    """
    scale = CONFIG["detection_scale"]
    small = cv2.resize(frame, (0, 0), fx=scale, fy=scale)
    gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    gray  = cv2.equalizeHist(gray)          # improve contrast for detection

    if model["backend"] == "haar":
        rects = model["detector"].detectMultiScale(
            gray,
            scaleFactor=CONFIG["scale_factor"],
            minNeighbors=CONFIG["min_neighbors"],
            minSize=CONFIG["min_face_size"],
            flags=cv2.CASCADE_SCALE_IMAGE,
        )
        if len(rects) == 0:
            return []
        # Scale coordinates back to original frame space
        inv = 1.0 / scale
        return [(int(x * inv), int(y * inv), int(w * inv), int(h * inv))
                for (x, y, w, h) in rects]

    # ── DNN detection (future) ───────────────────────────────────────────────
    # elif model["backend"] == "dnn":
    #     h_orig, w_orig = frame.shape[:2]
    #     blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)),
    #                                  1.0, (300, 300), (104, 177, 123))
    #     model["detector"].setInput(blob)
    #     detections = model["detector"].forward()
    #     faces = []
    #     for i in range(detections.shape[2]):
    #         confidence = detections[0, 0, i, 2]
    #         if confidence > 0.5:
    #             box = detections[0, 0, i, 3:7] * [w_orig, h_orig, w_orig, h_orig]
    #             x1, y1, x2, y2 = box.astype(int)
    #             faces.append((x1, y1, x2 - x1, y2 - y1))
    #     return faces
    # ────────────────────────────────────────────────────────────────────────

    return []


# ─────────────────────────────────────────────
#  Drawing
# ─────────────────────────────────────────────
def draw_bounding_boxes(frame, faces: list) -> None:
    """Draw bounding boxes and per-face labels on *frame* (in-place)."""
    color     = CONFIG["box_color"]
    thickness = CONFIG["box_thickness"]
    font      = CONFIG["font"]

    for idx, (x, y, w, h) in enumerate(faces):
        # Bounding rectangle
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, thickness)

        # Small label above each box
        label     = f"Face {idx + 1}"
        label_y   = max(y - 8, 16)
        cv2.putText(frame, label, (x, label_y),
                    font, 0.45, color, 1, cv2.LINE_AA)


def draw_hud(frame, face_count: int, fps: float) -> None:
    """Overlay HUD information (face count, FPS) on *frame* (in-place)."""
    font   = CONFIG["font"]
    h, w   = frame.shape[:2]
    margin = 10

    # Semi-transparent dark bar at the top
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, 36), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.45, frame, 0.55, 0, frame)

    # FPS — top-left
    cv2.putText(frame, f"FPS: {fps:5.1f}", (margin, 24),
                font, 0.7, (0, 255, 180), 2, cv2.LINE_AA)

    # Face count — top-right
    face_text = f"Faces: {face_count}"
    (tw, _), _ = cv2.getTextSize(face_text, font, 0.7, 2)
    cv2.putText(frame, face_text, (w - tw - margin, 24),
                font, 0.7, (0, 255, 180), 2, cv2.LINE_AA)

    # Exit hint — bottom-left
    cv2.putText(frame, "Press 'q' to quit", (margin, h - margin),
                font, 0.45, (180, 180, 180), 1, cv2.LINE_AA)


# ─────────────────────────────────────────────
#  Main loop
# ─────────────────────────────────────────────
def main_loop(cap: cv2.VideoCapture, model: dict) -> None:
    """
    Capture → detect → draw → display, until the user presses 'q'.

    FPS is computed as a rolling average over CONFIG['fps_smoothing'] frames
    to avoid jitter in the on-screen counter.
    """
    smoothing      = CONFIG["fps_smoothing"]
    frame_times    = []
    prev_time      = time.perf_counter()

    print("[Loop] Starting — press 'q' inside the window to quit.")

    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            print("[Loop] Failed to grab frame — stopping.")
            break

        # ── Detection ──────────────────────────────────────────────────────
        faces = detect_faces(frame, model)

        # ── Visualisation ──────────────────────────────────────────────────
        draw_bounding_boxes(frame, faces)

        # ── FPS bookkeeping ────────────────────────────────────────────────
        now         = time.perf_counter()
        frame_times.append(now - prev_time)
        prev_time   = now
        if len(frame_times) > smoothing:
            frame_times.pop(0)
        avg_dt = sum(frame_times) / len(frame_times)
        fps    = 1.0 / avg_dt if avg_dt > 0 else 0.0

        draw_hud(frame, len(faces), fps)

        # ── Display ────────────────────────────────────────────────────────
        cv2.imshow("Face Detection — Haar Cascade", frame)

        # ── Exit condition ─────────────────────────────────────────────────
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("[Loop] 'q' pressed — exiting.")
            break

    # ── Optional future hook: save_detected_faces(faces, frame) ───────────
    # ── Optional future hook: emotion_detection(faces, frame)   ───────────


# ─────────────────────────────────────────────
#  Entry point
# ─────────────────────────────────────────────
def main() -> None:
    cap   = None
    try:
        cap   = initialize_camera(CONFIG["camera_index"])
        model = load_model(MODEL_BACKEND)
        main_loop(cap, model)

    except RuntimeError as exc:
        print(f"[Error] Camera error: {exc}", file=sys.stderr)
        sys.exit(1)

    except FileNotFoundError as exc:
        print(f"[Error] Model not found: {exc}", file=sys.stderr)
        sys.exit(1)

    except KeyboardInterrupt:
        print("\n[Info] Interrupted by user.")

    finally:
        if cap is not None:
            cap.release()
        cv2.destroyAllWindows()
        print("[Done] Resources released.")


if __name__ == "__main__":
    main()

[Camera] Opened index=0  resolution=640x480
[Model] Loaded Haar cascade from d:\Data\Python\Various\venv\lib\site-packages\cv2\data\haarcascade_frontalface_default.xml
[Loop] Starting — press 'q' inside the window to quit.
[Loop] 'q' pressed — exiting.
[Done] Resources released.
